# LIF 交互式调参

调整恒定输入电流、发放阈值和膜时间常数，观察前 200 ms 的输入、膜电位、脉冲和发放率。松开滑块后重新仿真；请在支持 Jupyter Widgets 的环境中运行，GitHub 静态预览无法交互。

神经元和仿真函数复用 [task1_lif.ipynb](task1_lif.ipynb)；独立加载其模型单元，不运行原 Notebook 中的图表。


In [ ]:
import json
import sys
from pathlib import Path

import ipywidgets as widgets
import numpy as np
from IPython.display import display

# Notebook 可从项目根目录或 lif 目录启动。
lif_dir = Path.cwd() if Path.cwd().name == "lif" else Path.cwd() / "lif"
sys.path.insert(0, str(lif_dir.parent))
from lif.plot_lif import plot_summary  # noqa: E402

notebook = json.loads((lif_dir / "task1_lif.ipynb").read_text(encoding="utf-8"))
lif = {}
for cell in notebook["cells"]:
    if set(cell.get("metadata", {}).get("tags", [])) & {
        "lif-imports",
        "lif-defaults",
        "lif-core",
        "lif-rate",
    }:
        exec("".join(cell["source"]), lif)

In [ ]:
def explore(current_nA, V_th, tau_m):
    # 模型单元的全局变量保存在独立命名空间中。
    lif["V_th"], lif["tau_m"] = V_th, tau_m
    dt, duration_ms = lif["dt"], 200.0
    input_time_ms = np.arange(round(duration_ms / dt)) * dt
    current = np.full_like(input_time_ms, current_nA)
    voltage, spikes = lif["simulate"](current, duration_ms, dt)
    data = {
        "input_time_ms": input_time_ms,
        "time_ms": input_time_ms + dt,
        "current": current,
        "voltage": voltage,
        "spikes": spikes,
        "rate": lif["firing_rate"](spikes, dt),
    }
    plot_summary("Interactive input", data, lif["V_rest"], V_th)


controls = widgets.interactive(
    explore,
    current_nA=widgets.FloatSlider(
        value=2, min=0, max=4, step=0.1, description="I (nA)", continuous_update=False
    ),
    V_th=widgets.FloatSlider(
        value=-50,
        min=-60,
        max=-40,
        step=1,
        description="V_th (mV)",
        continuous_update=False,
    ),
    tau_m=widgets.FloatSlider(
        value=20,
        min=5,
        max=50,
        step=1,
        description="tau_m (ms)",
        continuous_update=False,
    ),
)
display(controls)